<a href="https://colab.research.google.com/github/2anoFIAP/AED-KPOP-CP-DATASCIENCE/blob/main/CP_AED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AED — shows de K-pop

## 1. Contextualização

### Pergunta central
Quais fatores estão associados à lucratividade da operação dos shows de K-pop?

### Contexto e limites
A base contém shows programados entre 2023 e 2025. Os nomes dos grupos são reais, mas os eventos e indicadores são simulados para fins educacionais.

Os resultados financeiros representam a operação de realização dos shows, não o lucro total dos grupos, artistas ou agências.

### Unidade de análise
Cada linha representa um show programado, inclusive quando cancelado. Também serão utilizadas agregações por turnê para avaliar o resultado da programação.

### Objetivo
Investigar relações entre características dos shows, contexto de mercado e desempenho financeiro, desenvolvendo pelo menos três insights e três ações vinculadas às evidências.

### Cuidados metodológicos
- Manter cancelamentos na avaliação financeira da operação.
- Não substituir automaticamente dados ausentes por zero.
- Distinguir lucro absoluto, margem e frequência de prejuízo.
- Considerar a dependência entre shows do mesmo grupo ou turnê.
- Interpretar associações sem presumir causalidade.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(
    "kpop_shows.csv",
    encoding="utf-8-sig",
    parse_dates=["data_show"]
)
df.head(5)

,id_show,dados_simulados,grupo,integrante_referencia,ano_estreia_grupo,data_show,ano,id_turne,ordem_show_turne,shows_planejados_turne,...,custo_marketing_usd,custo_produtos_usd,repasse_artistas_usd,comissao_promotor_usd,impostos_usd,rateio_abertura_turne_usd,receita_total_usd,custo_total_usd,lucro_operacional_usd,satisfacao_publico_0a10
0,K00001,True,TWICE,Jihyo,2015,2023-02-02,2023,T01-2023-1,1,8,...,54473.49,31141.86,54022.94,43156.84,32674.04,12555.79,523755.27,292376.31,231378.96,8.0
1,K00002,True,TWICE,Jihyo,2015,2023-02-05,2023,T01-2023-1,2,8,...,65081.42,91751.78,268334.08,198778.53,226494.08,12555.79,2357054.06,972198.04,1384856.02,8.5
2,K00003,True,TWICE,Jihyo,2015,2023-02-08,2023,T01-2023-1,3,8,...,86207.86,48788.28,179820.49,199192.56,68062.84,12555.79,1659454.20,672432.22,987021.98,7.2
3,K00004,True,TWICE,Jihyo,2015,2023-02-14,2023,T01-2023-1,4,8,...,50474.08,86804.44,310560.84,381878.22,137238.80,12555.79,2808116.01,1177750.74,1630365.27,8.4
4,K00005,True,TWICE,Jihyo,2015,2023-02-17,2023,T01-2023-1,5,8,...,40516.19,76202.43,226245.20,275082.71,162844.01,12555.79,1912873.37,934102.64,978770.73,7.5


## 2. Compreensão e inspeção dos dados

Antes de qualquer limpeza, vamos verificar a estrutura da base, os tipos das variáveis, a cobertura dos registros, possíveis duplicidades e os valores ausentes.

In [4]:
df_original = df.copy(deep=True)

print(f"Dimensões: {df.shape[0]:,} linhas e {df.shape[1]} colunas")
print(f"Grupos: {df['grupo'].nunique()}")
print(f"Turnês: {df['id_turne'].nunique()}")
print(f"Período: {df['data_show'].min():%d/%m/%Y}"
    f" a {df['data_show'].max():%d/%m/%Y}")

print("\nTipos e preenchimento das colunas:")
df.info()

print("\nStatus dos shows:")
display(df["status_show"].value_counts(dropna=False).rename_axis("status_show").to_frame("quantidade"))

print("\nVerificação de duplicidades:")
print(f"Linhas inteiramente duplicadas: {df.duplicated().sum()}")
print(f"IDs de show ausentes: {df['id_show'].isna().sum()}")
print(f"IDs de show duplicados: {df['id_show'].duplicated().sum()}")

ausentes = pd.DataFrame({
    "quantidade_ausente": df.isna().sum(),
    "percentual_ausente": df.isna().mean().mul(100)
})

ausentes = (
    ausentes.loc[ausentes["quantidade_ausente"] > 0]
    .sort_values("quantidade_ausente", ascending=False)
)

print("\nColunas com dados ausentes:")
display(ausentes.round(2))

Dimensões: 1,902 linhas e 48 colunas
Grupos: 20
Turnês: 180
Período: 20/01/2023 a 07/09/2025

Tipos e preenchimento das colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1902 entries, 0 to 1901
Data columns (total 48 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   id_show                          1902 non-null   object        
 1   dados_simulados                  1902 non-null   bool          
 2   grupo                            1902 non-null   object        
 3   integrante_referencia            1902 non-null   object        
 4   ano_estreia_grupo                1902 non-null   int64         
 5   data_show                        1902 non-null   datetime64[ns]
 6   ano                              1902 non-null   int64         
 7   id_turne                         1902 non-null   object        
 8   ordem_show_turne                 1902 non-null   int64         
 9   

,quantidade
status_show,
realizado,1878
cancelado,24



Verificação de duplicidades:
Linhas inteiramente duplicadas: 0
IDs de show ausentes: 0
IDs de show duplicados: 0

Colunas com dados ausentes:


,quantidade_ausente,percentual_ausente
intervalo_show_anterior_dias,180,9.46
satisfacao_publico_0a10,88,4.63
seguidores_locais_milhoes,67,3.52
streams_locais_30_dias_milhoes,58,3.05


### 2.1. Investigação dos dados ausentes

As ausências serão examinadas conforme o significado de cada variável.

- O intervalo desde o show anterior não se aplica à primeira data da turnê.
- A satisfação não se aplica aos shows cancelados; sua ausência em shows realizados representa falta de avaliação.
- Seguidores e streams possuem medições incompletas, que não devem ser interpretadas como zero.

Antes de definir tratamentos, verificaremos se os padrões observados são compatíveis com essas interpretações.

In [5]:
# 1. O intervalo ausente corresponde à primeira data da turnê?
primeiro_show = df["ordem_show_turne"].eq(1)
intervalo_ausente = df["intervalo_show_anterior_dias"].isna()

print("Intervalo desde o show anterior:")
display(
    pd.crosstab(
        primeiro_show,
        intervalo_ausente,
        rownames=["primeiro_show_da_turne"],
        colnames=["intervalo_ausente"],
        margins=True
    )
)

# 2. Como a ausência de satisfação se distribui por status?
resumo_satisfacao = (
    df.groupby("status_show")
    .agg(
        total_shows=("id_show", "size"),
        satisfacao_registrada=("satisfacao_publico_0a10", "count")
    )
)

resumo_satisfacao["satisfacao_ausente"] = (
    resumo_satisfacao["total_shows"]
    - resumo_satisfacao["satisfacao_registrada"]
)

resumo_satisfacao["percentual_ausente"] = (
    resumo_satisfacao["satisfacao_ausente"]
    / resumo_satisfacao["total_shows"]
    * 100
)

print("\nPreenchimento da satisfação por status:")
display(resumo_satisfacao.round(2))

# 3. Quantos shows têm ausência em pelo menos uma medida de popularidade?
colunas_popularidade = [
    "seguidores_locais_milhoes",
    "streams_locais_30_dias_milhoes"
]

popularidade_incompleta = df[colunas_popularidade].isna().any(axis=1)

print(
    "\nShows com seguidores ou streams ausentes:",
    popularidade_incompleta.sum()
)
print(
    "Shows com ambas as medidas ausentes:",
    df[colunas_popularidade].isna().all(axis=1).sum()
)

print("\nAusência de popularidade por região:")
display(
    df.assign(popularidade_incompleta=popularidade_incompleta)
    .groupby("regiao")
    .agg(
        total_shows=("id_show", "size"),
        popularidade_incompleta=("popularidade_incompleta", "sum"),
        percentual_incompleto=("popularidade_incompleta", "mean")
    )
    .assign(
        percentual_incompleto=lambda tabela:
            tabela["percentual_incompleto"] * 100
    )
    .round(2)
)

Intervalo desde o show anterior:


intervalo_ausente,False,True,All
primeiro_show_da_turne,,,
False,1722,0,1722
True,0,180,180
All,1722,180,1902



Preenchimento da satisfação por status:


,total_shows,satisfacao_registrada,satisfacao_ausente,percentual_ausente
status_show,,,,
cancelado,24,0,24,100.00
realizado,1878,1814,64,3.41



Shows com seguidores ou streams ausentes: 118
Shows com ambas as medidas ausentes: 7

Ausência de popularidade por região:


,total_shows,popularidade_incompleta,percentual_incompleto
regiao,,,
America Latina,429,26,6.06
America do Norte,518,28,5.41
Asia,443,28,6.32
Europa,512,36,7.03


### 2.2. Decisões sobre dados ausentes

A base principal será preservada, sem exclusão global de linhas ou
preenchimento automático de ausências.

- **Intervalo entre shows:** manter ausente nas primeiras datas das turnês,
  pois não existe evento anterior. Análises de intervalo considerarão
  somente shows com evento anterior.
- **Satisfação:** analisar shows realizados com nota registrada.
  Há 64 realizados sem avaliação (3,41%), o que limita a cobertura
  e pode introduzir viés caso esses shows sejam diferentes dos demais.
- **Popularidade:** manter seguidores e streams ausentes na base.
  Cada análise utilizará os registros com as medidas necessárias,
  informando o número de observações utilizado.

As proporções regionais de ausência, isoladamente, não permitem
determinar o mecanismo de falta dos dados.

In [6]:
# Padronizar a grafia das cidades, preservada em df_original

cidades_antes = df["cidade"].nunique()

df["cidade"] = (df["cidade"].str.strip().str.replace(r"\s+", " ", regex=True).str.casefold())

print(f"Cidades distintas antes: {cidades_antes}")
print(f"Cidades distintas depois: {df['cidade'].nunique()}")

# Verificações: sinalizar problemas sem excluir registros
verificacoes = {
    "Ano diferente do ano da data":
        df["ano"].ne(df["data_show"].dt.year),

    "Capacidade menor ou igual a zero":
        df["capacidade"].le(0),

    "Público pagante negativo":
        df["publico_pagante"].lt(0),

    "Cortesias negativas":
        df["cortesias"].lt(0),

    "Público total acima da capacidade":
        (df["publico_pagante"] + df["cortesias"]).gt(df["capacidade"]),

    "Câmbio menor ou igual a zero":
        df["cambio_moeda_local_por_usd"].le(0),

    "Preço efetivo negativo":
        df["preco_medio_efetivo_moeda_local"].lt(0),

    "Desconto fora de 0 a 100":
        ~df["desconto_medio_pct"].between(0, 100),

    "Satisfação registrada fora de 0 a 10":
        df["satisfacao_publico_0a10"].notna()
        & ~df["satisfacao_publico_0a10"].between(0, 10),

    "Intervalo registrado negativo":
        df["intervalo_show_anterior_dias"].lt(0),

    "Status não previsto no dicionário":
        ~df["status_show"].isin(["realizado", "cancelado"]),

    "Cancelado com público ou cortesias diferentes de zero":
        df["status_show"].eq("cancelado")
        & (
            df["publico_pagante"].ne(0)
            | df["cortesias"].ne(0)
        )
}

resumo_verificacoes = pd.DataFrame({
    "verificacao": list(verificacoes),
    "registros_sinalizados": [
        int(mascara.sum()) for mascara in verificacoes.values()
    ]
})

display(resumo_verificacoes)

# Exibir os registros sinalizados para investigação, se houver
for nome, mascara in verificacoes.items():
    if mascara.any():
        print(f"\n{nome}:")
        display(df.loc[mascara].head(10))

Cidades distintas antes: 31
Cidades distintas depois: 18


,verificacao,registros_sinalizados
0,Ano diferente do ano da data,0
1,Capacidade menor ou igual a zero,0
2,Público pagante negativo,0
3,Cortesias negativas,0
4,Público total acima da capacidade,0
5,Câmbio menor ou igual a zero,0
6,Preço efetivo negativo,0
7,Desconto fora de 0 a 100,0
8,Satisfação registrada fora de 0 a 10,0
9,Intervalo registrado negativo,0


### 2.3. Padronização e consistência

A padronização de espaços e letras maiúsculas/minúsculas reduziu
as grafias distintas de cidades de 31 para 18, preservando os registros.

Não foram encontradas violações nas 12 regras verificadas de datas,
capacidade, público, câmbio, preço, desconto, satisfação, intervalo
e status dos shows.

### 2.4. Validação financeira

Serão conferidas as identidades contábeis:

- Receita total = ingressos + produtos + patrocínio.
- Custo total = soma dos 11 componentes de custo.
- Lucro operacional = receita total − custo total.

Será utilizada uma tolerância de US$ 0,01 para diferenças numéricas.
Essa validação verifica consistência interna, não a veracidade externa
dos valores simulados.

In [7]:
colunas_receita = [
    "receita_ingressos_usd",
    "receita_produtos_usd",
    "receita_patrocinio_usd"
]

colunas_custo = [
    "custo_aluguel_usd",
    "custo_producao_usd",
    "custo_equipe_usd",
    "custo_transporte_usd",
    "custo_hospedagem_usd",
    "custo_marketing_usd",
    "custo_produtos_usd",
    "repasse_artistas_usd",
    "comissao_promotor_usd",
    "impostos_usd",
    "rateio_abertura_turne_usd"
]

# Exigir todos os componentes preenchidos para calcular cada soma
receita_recalculada = df[colunas_receita].sum(
    axis=1, min_count=len(colunas_receita)
)

custo_recalculado = df[colunas_custo].sum(
    axis=1, min_count=len(colunas_custo)
)

lucro_recalculado = receita_recalculada - custo_recalculado

comparacoes = {
    "Receita total": (
        receita_recalculada, df["receita_total_usd"]
    ),
    "Custo total": (
        custo_recalculado, df["custo_total_usd"]
    ),
    "Lucro operacional": (
        lucro_recalculado, df["lucro_operacional_usd"]
    )
}

resultados = []

for nome, (recalculado, informado) in comparacoes.items():
    diferenca = (recalculado - informado).abs()
    avaliavel = recalculado.notna() & informado.notna()

    resultados.append({
        "verificacao": nome,
        "registros_avaliados": int(avaliavel.sum()),
        "registros_nao_avaliaveis": int((~avaliavel).sum()),
        "divergencias_acima_de_1_centavo": int(
            (avaliavel & diferenca.gt(0.01)).sum()
        ),
        "maior_diferenca_absoluta_usd": diferenca.max()
    })

display(pd.DataFrame(resultados))

,verificacao,registros_avaliados,registros_nao_avaliaveis,divergencias_acima_de_1_centavo,maior_diferenca_absoluta_usd
0,Receita total,1902,0,0,9.313226e-10
1,Custo total,1902,0,0,4.656613e-10
2,Lucro operacional,1902,0,0,1.396984e-09
